In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

### ***Document Processing Tools***

In [7]:
from typing import List
from langchain_core.documents import Document

In [8]:
# Document Processing Tools and Class
class DocumentProcessor:
    @staticmethod
    def process_pdf(file_path: str) -> List[Document]:
        # Process PDF file using PyMuPDF
        loader = PyMuPDFLoader(file_path)
        return loader.load()
    
    @staticmethod
    def process_docx(file_path: str) -> List[Document]:
        # Process DOCX file
        loader = Docx2txtLoader(file_path)
        return loader.load()
    
    @staticmethod
    def process_txt(file_path: str) -> List[Document]:
        # Process TXT file
        loader = TextLoader(file_path)
        return loader.load()
    
    @staticmethod
    def process_url(url: str) -> List[Document]:
        # Process URL content
        loader = WebBaseLoader(url)
        return loader.load()

### ***Initialize models and tools***

In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [16]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\emon1\AppData\Local\Temp\ipykernel_8176\2537902316.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
from langchain_groq import ChatGroq

In [19]:
llm = ChatGroq(
    temperature=0.1,
    model_name="openai/gpt-oss-120b"
    )

In [20]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [21]:
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

### ***Text splitter for chunking documents***

In [23]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [24]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

### ***Define Agent States and Graph***

In [33]:
from typing import TypedDict
from typing import Optional
from typing import Literal
from typing import Dict

In [34]:
# Define the state for our graph
class AgentState(TypedDict):
    # Input from user
    input: str
    # File content (for uploads)
    file_content: Optional[str]
    # File type (pdf, docx, txt, url)
    file_type: Optional[Literal["pdf", "docx", "txt", "url"]]
    # Extracted document chunks
    chunks: Optional[List[str]]
    # Retrieved context
    context: Optional[List[str]]
    # Generated answer
    answer: Optional[str]
    # Conversation history
    history: List[Dict[str, str]]
    # Current agent
    current_agent: Optional[str]

In [36]:
from langchain_community.vectorstores import Chroma

In [37]:
vector_store = Chroma(embedding_function=embedding_model, persist_directory=None)

C:\Users\emon1\AppData\Local\Temp\ipykernel_8176\1903218020.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(embedding_function=embedding_model, persist_directory=None)


### ***Tool Router Agent***

In [38]:
# tool_router_agent function
def tool_router_agent(state: AgentState):

    if state.get("file_type") and state.get("file_content"):
        return {"current_agent": "ingestion_agent"}
    
    return {"current_agent": "planner_agent"}

### ***Ingestion Agent***

In [39]:
# ingestion_agent function
def ingestion_agent(state: AgentState):
    
    if not state.get("file_content") or not state.get("file_type"):
        return {"current_agent": "planner_agent"}
    
    # Process the document based on type
    file_type = state["file_type"]
    file_path = state["file_content"]
    
    if file_type == "pdf":
        documents = DocumentProcessor.process_pdf(file_path)
    elif file_type == "docx":
        documents = DocumentProcessor.process_docx(file_path)
    elif file_type == "txt":
        documents = DocumentProcessor.process_txt(file_path)
    elif file_type == "url":
        documents = DocumentProcessor.process_url(file_path)
    else:
        raise ValueError(f"Unsupported file type: {file_type}")
    
    # Split the documents into chunks
    chunks = text_splitter.split_documents(documents)
    
    # Store in vector DB
    if chunks:
        vector_store.add_documents(chunks)
    
    return {"current_agent": "planner_agent"}

### ***Planner Agent***

In [41]:
# planner_agent
def planner_agent(state: AgentState):
    
    if vector_store._collection.count() > 0:
        # if documents documents - use RAG path
        return {"next_agent": "retriever_agent"}
    else:
        # No documents - use fallback path
        return {"next_agent": "fallback_agent"}

In [42]:
# Retriever Agent
def retriever_agent(state: AgentState):

    question = state["input"]
    relevant_chunks = vector_store.similarity_search(question, k=3)
    context = [chunk.page_content for chunk in relevant_chunks]
    
    return {"context": context, "current_agent": "llm_answer_agent"}

### ***LLM Answer Agent***

In [74]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [61]:
def llm_answer_agent(state: AgentState):

    question = state["input"]
    context = state.get("context", [])
    history = state.get("history", [])
    
    # Prepare prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Answer the question based on the provided context."),
        *history,
        ("human", """
        Context:
        {context}
        
        Question: {question}
        
        Answer concisely and accurately. If you don't know, say so.
        """),
    ])
    
    # Format context
    formatted_context = "\n\n".join(context) if context else "No specific context provided."
    
    # Create chain
    chain = prompt | llm | StrOutputParser()
    
    # Generate answer
    answer = chain.invoke({
        "context": formatted_context,
        "question": question,
    })
    
    return {"answer": answer, "current_agent": "executor_agent"}

### ***Fallback Agent***

In [62]:
def fallback_agent(state: AgentState):
    
    question = state["input"]
    
    # Use Wikipedia as fallback
    try:
        result = wikipedia.run(question)
        answer = f"I couldn't find this in your documents, but here's what I found from Wikipedia:\n\n{result}"
    except:
        answer = "I couldn't find any relevant information to answer your question."
    
    return {"answer": answer, "current_agent": "executor_agent"}

### ***Executor Agent***

In [63]:
def executor_agent(state: AgentState):
    
    answer = state["answer"]
    question = state["input"]
    
    # Update conversation history
    history = state.get("history", [])
    history.extend([
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ])
    
    # Return final state
    return {
        "answer": answer,
        "history": history,
        "current_agent": END
    }

### ***Build the LangGraph Workflow***

In [64]:
from langgraph.graph import END
from langgraph.graph import StateGraph

In [65]:
# Define workflow edges
workflow = StateGraph(AgentState)

In [66]:
# Define nodes
workflow.add_node("tool_router_agent", tool_router_agent)
workflow.add_node("ingestion_agent", ingestion_agent)
workflow.add_node("planner_agent", planner_agent)
workflow.add_node("retriever_agent", retriever_agent)
workflow.add_node("llm_answer_agent", llm_answer_agent)
workflow.add_node("fallback_agent", fallback_agent)
workflow.add_node("executor_agent", executor_agent)

In [67]:
# Set entry point
workflow.set_entry_point("tool_router_agent")

In [68]:
# conditional edges from planner
workflow.add_conditional_edges(
    "planner_agent",
    lambda state: state.get("next_agent"),
    {
        "retriever_agent": "retriever_agent",
        "fallback_agent": "fallback_agent"
    }
)

In [69]:
# Define other edges
workflow.add_edge("tool_router_agent", "ingestion_agent")
workflow.add_edge("ingestion_agent", "planner_agent")
workflow.add_edge("retriever_agent", "llm_answer_agent")
workflow.add_edge("llm_answer_agent", "executor_agent")
workflow.add_edge("fallback_agent", "executor_agent")
workflow.add_edge("executor_agent", END)

In [70]:
# Compile the workflow
app = workflow.compile()

### ***Run the System***

In [71]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WebBaseLoader

In [72]:
def process_query(input_text: str, file_path: str = None, file_type: str = None):
    initial_state = {
        "input": input_text,
        "file_content": file_path,
        "file_type": file_type,
        "history": [],
    }
    
    # Run the workflow
    result = app.invoke(initial_state)
    
    return result["answer"]

### ***Example Usage***

In [75]:
# Example 1: Using a PDF file
pdf_path = "../data/Md Hasan Imon - (AI & ML) Resume.pdf"

# Process the PDF
print("Processing PDF...")
process_query("Please analyze this document", file_path=pdf_path, file_type="pdf")

Processing PDF...


'**Document Summary**\n\n- **Goal:** Provide a semantic‑search solution that can pull knowledge from any type of document (PDF, Word, HTML, etc.) with high accuracy and the ability to scale to large corpora.  \n- **Key Features**  \n  1. **Dynamic Tool Routing & Fallback:**  \n     - Automatically chooses the most trustworthy source for a query—either a Retrieval‑Augmented Generation (RAG) index, a live web‑search API, or an internal memory store.  \n     - Guarantees “trustworthy and resilient” answers by switching to a fallback when the primary source is insufficient or unreliable.  \n  2. **User Interface + CI/CD:**  \n     - A front‑end that lets users interact with documents of any format.  \n     - Continuous Integration/Continuous Deployment pipelines keep the UI and backend in sync, enabling rapid updates and reliable releases.  \n\n**Strengths**\n\n- **Adaptability:** The routing logic makes the system robust across varying query contexts and data availability.  \n- **Scalabil

In [76]:
answer = process_query("What is the main topic of this document?")
print("Answer:", answer)

Answer: The document is a résumé highlighting the author’s skills and experience in NLP, large language models, AI agents, vector databases, and cloud deployment.


In [77]:
answer = process_query("who is Imon?")
print("Answer:", answer)

Answer: Imon is MD Hasan Imon, an AI & ML Engineer based in Savar, Dhaka, Bangladesh.


In [78]:
# Example 2: Process a URL
url = "https://en.wikipedia.org/wiki/Large_language_model"
process_query("Please analyze this webpage", file_path=url, file_type="url")

'**Webpage Overview**\n\n- **Source:** Wikipedia article titled *Large language model* (revision\u202f1312897988, last edited\u202f23\u202fSept\u202f2025).  \n- **License:** Creative Commons Attribution‑ShareAlike\u202f4.0 (standard Wikipedia licensing).  \n- **Structure:**  \n  - Bibliographic references (e.g., Sutherland\u202f2024, Paaß\u202f&\u202fGiesselbach\u202f2022, Dodge\u202fet\u202fal.\u202f2021).  \n  - Category tags (e.g., “Large language models”, “Deep learning”, “Natural language processing”).  \n  - Hidden maintenance categories indicating editorial issues (unsourced statements, potentially dated statements from 2025, etc.).  \n  - Footer with licensing notice and site‑wide terms.\n\n- **Content excerpt:**  \n  - Discusses public LLM applications (ChatGPT, Claude) and their built‑in safety filters.  \n  - Highlights that bypassing these controls remains a problem, citing a 2023 study on safety‑system evasion.  \n  - Summarizes a 2025 report by the American Sunlight Proje

In [79]:
answer = process_query("What are large language models?")
print("Answer:", answer)

Answer: Large language models (LLMs) are massive neural‑network systems—most often transformer‑based—that are trained on huge corpora of text (ranging from billions to trillions of words) and contain billions or even trillions of parameters.  By learning statistical patterns of language, they can generate, complete, translate, and reason about natural‑language text across a wide variety of tasks.


In [80]:
answer = process_query("Who invented the telephone?")
print("Answer:", answer)

Answer: Alexander Graham Bell is credited with inventing the telephone, receiving the first US patent for it in 1876.
